# Vesuvius Autoresearch: Night Shift Analysis

Plots and summaries from the autoresearch loop's Night Shift sprints. Inputs:

- `results.tsv` — per-experiment record. Schema this notebook assumes: `timestamp`, `val_bpb`, `throughput_Mvps`, `num_params_M`. (Note: `notebooks/analysis.ipynb` assumes a different schema; whichever notebook last regenerated the file determines the actual columns. If a cell errors on `KeyError`, the schema has drifted.)
- `reports/figures/training_samples/*.png` — training-data audit images.
- `predictions/*.png` — most recent inference outputs.

Caveat on validation framing: in this repo's current configuration the `val_uri` is `local_data/PHercParis2Fr143/surface_volume.zarr` — a held-out fragment from the same scroll as the training pool. That is fragment-level validation, not cross-scroll. Cross-scroll transfer to Scroll 2 / Scroll 3 is the active research target, not a measured result.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import glob
from PIL import Image
import datetime
import seaborn as sns

# Set professional plotting style
plt.style.use('seaborn-v0_8-muted')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.family'] = 'sans-serif'

def load_results(path='results.tsv'):
    if not os.path.exists(path):
        return pd.DataFrame()
    df = pd.read_csv(path, sep='\t')
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df.sort_values('timestamp')

df = load_results()
print(f"SUCCESS: Loaded {len(df)} experimental data points.")
if not df.empty:
    display(df.tail(3))

## 1. Longitudinal Metric Tracking

We track the `val_bpb` metric, which represents **1.0 - Validation Dice Score** (Dice Loss). This measures how well the model's ink predictions match manual ground truth on a scroll it has never seen before.

In [ ]:
if not df.empty:
    fig, ax1 = plt.subplots(figsize=(15, 8))

    # Primary metric: val_bpb (1 - centerline Dice on the held-out fragment)
    sns.lineplot(data=df, x='timestamp', y='val_bpb', marker='o', color='#34495e', linewidth=3, ax=ax1, label='val_bpb (lower is better)')
    ax1.set_xlabel('Experiment Timeline', fontweight='bold')
    ax1.set_ylabel('val_bpb (held-out fragment, lower is better)', fontweight='bold', color='#34495e')
    ax1.set_yscale('log')

    # Secondary metric: throughput (Mvps)
    ax2 = ax1.twinx()
    sns.lineplot(data=df, x='timestamp', y='throughput_Mvps', marker='x', color='#e74c3c', linestyle='--', alpha=0.6, ax=ax2, label='Throughput (Mvps)')
    ax2.set_ylabel('Throughput (Mvps)', fontweight='bold', color='#e74c3c')

    plt.title('Autoresearch progress: val_bpb running min vs throughput', fontsize=16, pad=20)
    ax1.grid(True, which='both', ls='-', alpha=0.2)
    fig.tight_layout()
    plt.show()
else:
    print('INFO: No experimental data available yet. Run an autoresearch sprint or check that results.tsv exists.')

## 2. Training Data Audit

It is critical to verify that the model is receiving high-quality, aligned patches. The following visualization shows random samples from the active training set (CT Slice | Ground Truth | Overlay).

In [ ]:
train_sample_images = sorted(glob.glob('reports/figures/training_samples/*.png'))
if train_sample_images:
    latest_train_sample = train_sample_images[-1]
    print(f"AUDIT: Displaying training samples: {latest_train_sample}")
    img = Image.open(latest_train_sample)
    plt.figure(figsize=(16, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print("WARNING: No training samples found. Run scripts/visualize_training_data.py")

## 3. Qualitative Visual Validation

Beyond metrics, we visually audit the **Ink Detection Overlays**. Professional submissions must show clear spatial context (CT scan layers + Fiber signals + Ink maps).

In [ ]:
pred_images = sorted(glob.glob('predictions/*.png'))
if pred_images:
    latest_pred = pred_images[-1]
    print(f"AUDIT: Displaying most recent model inference: {latest_pred}")
    img = Image.open(latest_pred)
    plt.figure(figsize=(16, 9))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Technical Discovery Snapshot: CT | Fiber | Ink Overlay', pad=15)
    plt.show()
else:
    print("WARNING: No discovery images found. Verify prediction pipeline output.")

## 4. Top-5 by val_bpb

The five configurations that achieved the lowest `val_bpb` on the held-out fragment (`PHercParis2Fr143`). This is in-distribution validation against a fragment from the same scroll as the training pool, not cross-scroll transfer.

In [ ]:
if not df.empty:
    # Select top 5 and format
    leaderboard = df.nsmallest(5, 'val_bpb').copy()
    leaderboard['Improvement (%)'] = ((df.iloc[0]['val_bpb'] - leaderboard['val_bpb']) / df.iloc[0]['val_bpb'] * 100)
    
    print("Top Discovery Milestones (Ranked by Dice Loss):")
    styled_lb = leaderboard[['timestamp', 'val_bpb', 'throughput_Mvps', 'num_params_M', 'Improvement (%)']].style.format({
        'val_bpb': '{:.6f}',
        'throughput_Mvps': '{:.2f}',
        'num_params_M': '{:.2f}',
        'Improvement (%)': '{:+.2f}%'
    })
    display(styled_lb)
    
    best_dice = (1 - df['val_bpb'].min()) * 100
    print(f"\nPeak Validation Dice Score: {best_dice:.2f}%")

---
*Regenerate by re-running all cells against the latest `results.tsv`. If a cell errors with `KeyError` on a column, the TSV schema has drifted from what this notebook expects — see the intro cell for the assumed schema.*